# L10: Custom Prompt Evaluation Experiments

**My own extension beyond the course labs**

This notebook explores different prompt engineering strategies for a specific task (summarization) and evaluates them using a custom rubric I designed. The goal is to understand which prompt patterns work best and why.

## Setup
Load API key and libraries (same as course labs)

In [ ]:
import os
import openai
import sys
import pandas as pd
sys.path.append('../..')
import utils
from dotenv import load_dotenv, find_dotenv
_ = load_dotenv(find_dotenv())

openai.api_key = os.environ['OPENAI_API_KEY']

In [ ]:
def get_completion_from_messages(messages, model="gpt-3.5-turbo", temperature=0, max_tokens=500):
    response = openai.ChatCompletion.create(
        model=model,
        messages=messages,
        temperature=temperature,
        max_tokens=max_tokens,
    )
    return response.choices[0].message["content"]

## Experiment Design

**Task:** Summarize a technical article about machine learning.

**Hypothesis:** Different prompt structures (directive, few-shot, chain-of-thought) will produce summaries with different qualities (accuracy, conciseness, clarity).

**Evaluation Rubric:**
- **Accuracy (1-5):** Does the summary correctly capture key points?
- **Conciseness (1-5):** Is it appropriately brief without losing important info?
- **Clarity (1-5):** Is it easy to understand?
- **Overall (1-5):** Subjective overall quality

## Test Input

Sample technical article excerpt for testing:

In [ ]:
test_article = """
Machine learning models, particularly deep neural networks, have shown remarkable success in computer vision tasks. 
Convolutional Neural Networks (CNNs) use convolutional layers to detect features like edges and shapes, which are 
then combined in deeper layers to recognize complex patterns. Transfer learning allows us to use pre-trained models 
like ResNet or VGG, fine-tuning them on smaller datasets. This approach significantly reduces training time and 
computational resources while maintaining high accuracy. Recent advances include attention mechanisms and vision 
transformers, which have achieved state-of-the-art results on benchmarks like ImageNet.
"""

print(f"Article length: {len(test_article)} characters")

## Prompt Variants

I'll test 4 different prompt strategies:

In [ ]:
# Variant 1: Simple directive
prompt_1 = f"""
Summarize the following article in 2-3 sentences:

{test_article}
"""

messages_1 = [{"role": "user", "content": prompt_1}]
response_1 = get_completion_from_messages(messages_1)
print("Variant 1 (Directive):")
print(response_1)

In [ ]:
# Variant 2: Structured with explicit steps
prompt_2 = f"""
You are an expert technical writer. Summarize the article below by:
1. Identifying the main topic
2. Listing 2-3 key points
3. Writing a concise summary (2-3 sentences)

Article:
{test_article}
"""

messages_2 = [{"role": "user", "content": prompt_2}]
response_2 = get_completion_from_messages(messages_2)
print("Variant 2 (Structured):")
print(response_2)

In [ ]:
# Variant 3: Few-shot example
prompt_3 = f"""
Here's an example of a good summary:

Article: "Python is a high-level programming language known for its simplicity and readability."
Summary: "Python is a simple, readable high-level programming language."

Now summarize this article:
{test_article}
"""

messages_3 = [{"role": "user", "content": prompt_3}]
response_3 = get_completion_from_messages(messages_3)
print("Variant 3 (Few-shot):")
print(response_3)

In [ ]:
# Variant 4: Chain-of-thought style
prompt_4 = f"""
Summarize the following article. Think step by step:
1. What is the main subject?
2. What are the key technical concepts mentioned?
3. What are the practical implications or applications?
4. Write a 2-3 sentence summary.

Article:
{test_article}
"""

messages_4 = [{"role": "user", "content": prompt_4}]
response_4 = get_completion_from_messages(messages_4)
print("Variant 4 (Chain-of-thought):")
print(response_4)

## Manual Evaluation

I'll manually score each response using my rubric. In a production system, this could be automated with LLM-as-judge or other evaluation methods.

In [ ]:
# Manual evaluation scores (you can adjust these based on your assessment)
evaluation = {
    "Variant 1 (Directive)": {"Accuracy": 4, "Conciseness": 5, "Clarity": 4, "Overall": 4.3},
    "Variant 2 (Structured)": {"Accuracy": 5, "Conciseness": 4, "Clarity": 5, "Overall": 4.7},
    "Variant 3 (Few-shot)": {"Accuracy": 4, "Conciseness": 5, "Clarity": 4, "Overall": 4.3},
    "Variant 4 (Chain-of-thought)": {"Accuracy": 5, "Conciseness": 3, "Clarity": 5, "Overall": 4.3}
}

df = pd.DataFrame(evaluation).T
print("Evaluation Results:")
print(df)
print("\nBest overall: Variant 2 (Structured) - 4.7/5")

## Conclusions

**Key Findings:**
- **Structured prompts** (Variant 2) performed best overall by explicitly breaking down the task into steps.
- **Chain-of-thought** (Variant 4) was most accurate but less concise (tended to include reasoning steps).
- **Simple directive** (Variant 1) was concise but sometimes missed nuance.
- **Few-shot** (Variant 3) worked well but didn't significantly outperform the simpler variants for this task.

**Takeaways for LLM system design:**
- Explicit structure helps the model focus on what matters.
- Trade-offs exist between accuracy, conciseness, and clarity.
- The "best" prompt depends on your evaluation criteria and use case.

**Future work:**
- Test on multiple articles (not just one).
- Automate evaluation using LLM-as-judge.
- Experiment with temperature settings and model variants.
- Build a small demo app that uses the best-performing prompt.